# MuscleMap WB — Asian Dataset Evaluation (Thigh, Water)

Computes per-muscle Dice / Hausdorff metrics for MuscleMap WB segmentations on the **MRI_data_asian** thigh water images.

- **Predictions**: `asian_segs_water/{subject}/Thigh/*_dseg*`
- **Ground truth**: `MRI_data_asian/MRI_data/{subject}/Thigh/mask_muscles.nii.gz` (labels 1–13)

**The Asian GT is unilateral** — each label covers only one side of the thigh; the other side is background.  
L and R model predictions are therefore evaluated **separately** against the same GT label.  
The side with meaningful Dice is the labeled side; the other will show near-zero Dice / elevated false positives.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', 'asian_segs_water')
DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
RESULT_DIR = 'results_asian_water'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, asian_gt_label, mm_labels)
# GT is UNILATERAL — L and R are evaluated separately against the same GT label.
# The correct side will show meaningful Dice; the other near-zero / high FP.
# biceps_femoris: model has Long Head + Short Head per side — kept separate too.
# gluteus_maximus: not in the WB model → None (all-zero prediction).
MUSCLES = [
    ('rectus_femoris_L',          1,  [7131]),
    ('rectus_femoris_R',          1,  [7132]),
    ('vastus_lateralis_L',        2,  [7101]),
    ('vastus_lateralis_R',        2,  [7102]),
    ('vastus_intermedius_L',      3,  [7111]),
    ('vastus_intermedius_R',      3,  [7112]),
    ('vastus_medialis_L',         4,  [7121]),
    ('vastus_medialis_R',         4,  [7122]),
    ('sartorius_L',               5,  [7141]),
    ('sartorius_R',               5,  [7142]),
    ('gracilis_L',                6,  [7151]),
    ('gracilis_R',                6,  [7152]),
    ('biceps_femoris_L',          7,  [7181, 7191]),  # long + short head, left
    ('biceps_femoris_R',          7,  [7182, 7192]),  # long + short head, right
    ('semitendinosus_L',          8,  [7171]),
    ('semitendinosus_R',          8,  [7172]),
    ('semimembranosus_L',         9,  [7161]),
    ('semimembranosus_R',         9,  [7162]),
    ('adductor_brevis_L',         10, [7221]),
    ('adductor_brevis_R',         10, [7222]),
    ('adductor_longus_L',         11, [7211]),
    ('adductor_longus_R',         11, [7212]),
    ('adductor_magnus_L',         12, [7201]),
    ('adductor_magnus_R',         12, [7202]),
    ('gluteus_maximus',           13, None),
]

print(f'SEG_DIR   : {os.path.abspath(SEG_DIR)}')
print(f'DATA_ROOT : {os.path.abspath(DATA_ROOT)}')
print(f'RESULT_DIR: {os.path.abspath(RESULT_DIR)}')

In [ ]:
# ── Discover Thigh segmentation files ─────────────────────────────────────────

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, '*', 'Thigh', '*_dseg*')))
print(f'Found {len(seg_files)} Thigh segmentation files:')
for p in seg_files:
    print(' ', p)

In [ ]:
def evaluate_muscle(muscle_name, gt_label, mm_label_l, mm_label_r,
                    seg_files, result_dir):
    """
    For each seg file, load the GT mask_muscles.nii.gz and the prediction,
    merge L+R MuscleMap labels, and compute overlap metrics.
    """
    results = []
    for seg_path in seg_files:
        # Parse subject from path: .../asian_segs_water/{subject}/Thigh/*
        parts   = seg_path.replace('\\', '/').split('/')
        subject = parts[-3]   # one level above 'Thigh'

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_sitk = sitk.ReadImage(seg_path)
        seg_raw  = sitk.GetArrayFromImage(seg_sitk)
        # Merge bilateral MuscleMap labels to match the GT's single label
        pred_arr = ((seg_raw == mm_label_l) | (seg_raw == mm_label_r)).astype(np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {subject}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(seg_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_musclemap_wb_asian_water.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────

dfs = {}
for muscle_name, gt_label, mm_label_l, mm_label_r in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, MM={mm_label_l}+{mm_label_r}) ──')
    dfs[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, mm_label_l, mm_label_r,
        seg_files, RESULT_DIR,
    )

print('\nDone.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

from IPython.display import display

summary_rows = []
for muscle_name, df in dfs.items():
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':          muscle_name,
        'n':               len(df),
        'dice_mean':       df[dice_col].mean(),
        'dice_std':        df[dice_col].std(),
        'hausdorff_mean':  df[hd_col].mean(),
        'hausdorff_std':   df[hd_col].std(),
    })

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, 'summary_musclemap_wb_asian_water.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))